# Chronoscope Raft chart demo

This notebook installs Chronoscope from PyPI, downloads the demo config and trace from the matching GitHub release, builds a temporary database, and displays its Raft timelines. It does not clone or check out the repository. Run the cells from top to bottom. The source trace is not modified.

> **Colab:** the first run installs the interactive backend and restarts the kernel once. After it reconnects, choose **Runtime → Run all** again.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile
from urllib.request import urlretrieve

CHRONOSCOPE_VERSION = "0.0.5.dev0"
RELEASE_TAG = f"v{CHRONOSCOPE_VERSION}"
RAW_BASE_URL = f"https://raw.githubusercontent.com/just-now/chronoscope/{RELEASE_TAG}"
IN_COLAB = "google.colab" in sys.modules

data_dir = (
    Path("/content/chronoscope-demo-data")
    if IN_COLAB
    else Path(tempfile.gettempdir()) / "chronoscope-demo-data"
)
data_dir.mkdir(parents=True, exist_ok=True)

config = data_dir / "raft_chronoscope.yaml"
trace = data_dir / "gateway_failover_many_transactions_trace.txt"
for destination, source_path in (
    (config, "test/raft_chronoscope.yaml"),
    (trace, "examples/data/gateway_failover_many_transactions_trace.txt"),
):
    if not destination.exists():
        temporary = destination.with_suffix(destination.suffix + ".download")
        urlretrieve(f"{RAW_BASE_URL}/{source_path}", temporary)
        temporary.replace(destination)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        f"Chronoscope=={CHRONOSCOPE_VERSION}", "ipympl",
    ],
    check=True,
)

if IN_COLAB:
    restart_marker = Path(f"/content/.chronoscope_{CHRONOSCOPE_VERSION}_ready")
    if not restart_marker.exists():
        restart_marker.touch()
        print("Packages installed. Restarting the kernel; then run all cells again.")
        get_ipython().kernel.do_shutdown(restart=True)

print(f"Using Chronoscope {CHRONOSCOPE_VERSION} from PyPI")
print(f"Demo assets downloaded to {data_dir}")

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

get_ipython().run_line_magic("matplotlib", "widget")

## Build the demo database

The database lives in a temporary directory and is removed when the notebook kernel stops.

In [ ]:
from chronoscope import db, db_options
from chronoscope.parser import parser

demo_directory = tempfile.TemporaryDirectory()
demo_db = Path(demo_directory.name) / "raft_demo.db"

db.open(str(demo_db), db_options, create=True)
try:
    db.load(parser(str(config)), str(trace))
    db.mkidx()
    raft_ids = [
        row[0]
        for row in (
            db.state_machine
            .select(db.state_machine.id)
            .where(db.state_machine.type == "raft")
            .tuples()
        )
    ]
    if not raft_ids:
        raise RuntimeError("The trace contains no Raft state machines.")

    top_sm_id = (
        db.state_machine
        .select(db.state_machine.id)
        .order_by(db.state_machine.id)
        .scalar()
        - 1
    )
    with db.db.atomic():
        db.state_machine.create(id=top_sm_id, name="top", type="top")
        db.state_machine_relation.insert_many([
            {
                "from_sm_id": top_sm_id,
                "to_sm_id": raft_id,
                "relation": "top-to-raft",
            }
            for raft_id in raft_ids
        ]).execute()
finally:
    db.close()

print(f"Loaded {len(raft_ids)} Raft state machines; chart root: {top_sm_id:#x}")

<div style="font-size: 24px; line-height: 1.5;">
  <h1 style="font-size: 46px;">⚠️ Before you start: read this section</h1>
  <h2 style="font-size: 36px;">Display the Raft timelines</h2>
  <p>The toolbar supports pan, zoom, and saving the chart. <strong>Click the chart once</strong> to give it keyboard focus before using a shortcut.</p>
  <table style="font-size: 22px; width: 100%;">
    <thead><tr><th style="text-align: left;">Key</th><th style="text-align: left;">Action</th></tr></thead>
    <tbody>
      <tr><td><code>o</code></td><td>Toggle rectangle zoom, then drag over the area to enlarge.</td></tr>
      <tr><td><code>←</code>, <code>c</code>, or <code>Backspace</code></td><td>Go back to the previous zoom or view.</td></tr>
      <tr><td><code>→</code> or <code>v</code></td><td>Go forward to the next view.</td></tr>
      <tr><td><code>p</code></td><td>Toggle pan mode. Drag with the left mouse button to pan; drag with the right button to zoom.</td></tr>
      <tr><td><code>h</code>, <code>r</code>, or <code>Home</code></td><td>Reset to the original view.</td></tr>
      <tr><td><code>e</code>, then click</td><td>Place a measurement marker. Repeat for the second endpoint.</td></tr>
      <tr><td><code>d</code></td><td>Remove the latest measurement marker.</td></tr>
      <tr><td><code>s</code> or <code>Ctrl+S</code></td><td>Save the chart.</td></tr>
    </tbody>
  </table>
  <p><strong>To measure an interval, repeat <code>e</code> → click for each of its two endpoints.</strong></p>
</div>

In [ ]:
from chronoscope import chart

db.open(str(demo_db), db_options)
try:
    chart.plot(top_sm_id, figsize=(12, 8))
finally:
    db.close()

## Try another trace

Replace `config` and `trace` above with paths to files using the same Chronoscope formats, then rerun the last two code cells. In Colab, files can be uploaded through the Files panel.